In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
from pyspark.sql import Window, functions as F


In [2]:
S3_BUCKET = "datalake-teste2"
S3_BRONZE = f"s3a://{S3_BUCKET}/bronze"
S3_SILVER = f"s3a://{S3_BUCKET}/silver"
S3_GOLD = f"s3a://{S3_BUCKET}/gold"

SPARK_MASTER = "local[*]"
SPARK_APP_NAME = "medallion-pipeline"

print(f"Bronze:  {S3_BRONZE}")
print(f"Silver:  {S3_SILVER}")
print(f"Gold:    {S3_GOLD}")

Bronze:  s3a://datalake-teste2/bronze
Silver:  s3a://datalake-teste2/silver
Gold:    s3a://datalake-teste2/gold


In [3]:
spark = SparkSession.builder \
    .appName(SPARK_APP_NAME) \
    .master(SPARK_MASTER) \
    .config("spark.hadoop.fs.s3a.endpoint", "http://localhost:4566") \
    .config("spark.hadoop.fs.s3a.access.key", "test") \
    .config("spark.hadoop.fs.s3a.secret.key", "test") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.hadoop.fs.s3a.connection.ssl.enabled", "false") \
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.2,com.amazonaws:aws-java-sdk-bundle:1.12.261") \
    .getOrCreate()

26/09/03 01:07:55 WARN Utils: Your hostname, hugo resolves to a loopback address: 127.0.1.1; using 10.159.141.203 instead (on interface wlp2s0)
26/09/03 01:07:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/hugo/Desktop/Project/venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/hugo/.ivy2/cache
The jars for the packages stored in: /home/hugo/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ee85a307-e597-479c-8122-6cff5c247331;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.2 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.261 in central
:: resolution report :: resolve 333ms :: artifacts dl 14ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.261 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.2 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	:: evicted modules:
	com.amazonaws#aws-java-sdk-bundle;1.11.1026 by [com.amazonaws#aws-java-sdk-bundle;1.12.261] in [default]
	------------------------------------------------------------------

In [4]:
df = spark.read.parquet(f"{S3_SILVER}/eventos_unificados")
print(f"\n📥 {df.count()} eventos lidos de eventos_unificados")

26/09/03 01:08:02 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties



📥 36 eventos lidos de eventos_unificados


In [5]:
linha_do_tempo = Window.partitionBy("purchase_id").orderBy(
        F.col("transaction_datetime").asc(),
        F.col("hash_evento").asc(),
    ).rowsBetween(Window.unboundedPreceding, Window.currentRow)

linha_do_tempo

In [6]:
# Colunas que identificam o evento, presentes em todas as fontes.
CHAVES = ["transaction_datetime", "transaction_date", "purchase_id", "origem_evento"]


COLUNAS_POR_FONTE = {
    "purchase": ["buyer_id", "prod_item_id", "order_date", "release_date", "producer_id"],
    "product_item": ["product_id", "item_quantity", "purchase_value"],
    "purchase_extra_info": ["subsidiary"],
}

# Ordem estável das colunas: chaves primeiro, depois um bloco por fonte.
COLUNAS = CHAVES + [c for cols in COLUNAS_POR_FONTE.values() for c in cols]


for fonte, colunas in COLUNAS_POR_FONTE.items():
        bloco = F.when(F.col("origem_evento") == fonte, F.struct(*colunas))
        df = df.withColumn(
            f"_ultimo_{fonte}", F.last(bloco, ignorenulls=True).over(linha_do_tempo)
        )

In [7]:
# Expande os structs de volta em colunas planas. Só depois de TODOS estarem
    # calculados, senão sobrescrever uma coluna afetaria o struct seguinte.
for fonte, colunas in COLUNAS_POR_FONTE.items():
    for coluna in colunas:
        df = df.withColumn(coluna, F.col(f"_ultimo_{fonte}.{coluna}"))
    df = df.drop(f"_ultimo_{fonte}")


In [8]:
# --- 4.2 Colapsa para o grão diário -------------------------------------
# Uma linha por (purchase_id, transaction_date): a última do dia, que já
# carrega o estado consolidado das 3 fontes até aquele momento.

dia = Window.partitionBy("purchase_id", "transaction_date")


In [9]:
# Quais fontes dispararam evento nesse dia — rastreabilidade diária.
    # Substitui origem_evento, que após o colapso só diria a última.
df = df.withColumn(
        "fontes_no_dia", F.sort_array(F.collect_set("origem_evento").over(dia))
    )

df.show()

+--------------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+----------------+--------------------+
|transaction_datetime|purchase_id|      origem_evento|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|   subsidiary|         hash_evento|transaction_date|       fontes_no_dia|
+--------------------+-----------+-------------------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+----------------+--------------------+
| 2023-01-20 22:00:00|         55|           purchase|   15947|           5|2023-01-20|  2023-01-20|     852852|      NULL|         NULL|          NULL|         NULL|68e913ce61509472e...|      2023-01-20|[product_item, pu...|
| 2023-01-20 22:02:00|         55|       product_item|   15947|           5|2023-01-20|  2023-01

In [10]:
ultimo_do_dia = dia.orderBy(F.col("transaction_datetime").desc(), F.col("hash_evento").desc(),)

In [11]:
df = (
    df.withColumn("_rn", F.row_number().over(ultimo_do_dia))
    .filter(F.col("_rn") == 1)
    .drop("_rn", "origem_evento")
)

df.show()

+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+----------------+--------------------+
|transaction_datetime|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|   subsidiary|         hash_evento|transaction_date|       fontes_no_dia|
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+--------------------+----------------+--------------------+
| 2023-01-20 22:02:00|         55|   15947|           5|2023-01-20|  2023-01-20|     852852|    696969|           10|         50.00|         NULL|56bfb324c970e2e7b...|      2023-01-20|[product_item, pu...|
| 2023-01-23 00:05:00|         55|   15947|           5|2023-01-20|  2023-01-20|     852852|    696969|           10|         50.00|     nacional|0ed2b358ec2fe6113...|      202

In [15]:
df = (
    df.drop("hash_evento").orderBy("purchase_id", "transaction_date"))


df.show()

+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+----------------+--------------------+
|transaction_datetime|purchase_id|buyer_id|prod_item_id|order_date|release_date|producer_id|product_id|item_quantity|purchase_value|   subsidiary|transaction_date|       fontes_no_dia|
+--------------------+-----------+--------+------------+----------+------------+-----------+----------+-------------+--------------+-------------+----------------+--------------------+
| 2023-01-20 22:02:00|         55|   15947|           5|2023-01-20|  2023-01-20|     852852|    696969|           10|         50.00|         NULL|      2023-01-20|[product_item, pu...|
| 2023-01-23 00:05:00|         55|   15947|           5|2023-01-20|  2023-01-20|     852852|    696969|           10|         50.00|     nacional|      2023-01-23|[purchase_extra_i...|
| 2023-02-05 10:00:00|         55|  160001|           5|2023-01-20|  2023-0

In [16]:
df.write.mode("overwrite").partitionBy("transaction_date").parquet(f"{S3_SILVER}/purchase_diario")